In [ ]:
import polars as pl
import warnings
import os 

warnings.filterwarnings("ignore")
warnings.simplefilter(action='ignore', category=FutureWarning)
os.environ['POLARS_WARN_UNSTABLE'] = '0'

In [ ]:
# Configure Polars to show all rows and make it scrollable
pl.Config.set_tbl_rows(-1)  # Show all rows
pl.Config.set_tbl_cols(-1)  # Show all columns
pl.Config.set_tbl_width_chars(1000)  # Set wider table width

In [ ]:
# Load all data files using Polars
print("Loading datasets with Polars...")
# Additional data sources
previous_df = pl.read_csv('../data/previous_application.csv')
print(f"Previous application data shape: {previous_df.shape}")

In [ ]:
# Print the first few rows to verify loading
previous_df.head()

In [ ]:
# === PREVIOUS APPLICATION COMPREHENSIVE EDA ===
print("=== PREVIOUS APPLICATION DETAILED ANALYSIS ===")

print(f"Dataset shape: {previous_df.shape}")
print(f"Columns: {list(previous_df.columns)}")

# Basic statistics
print(f"\nBasic Statistics:")
print(f"Unique current loans: {previous_df['SK_ID_CURR'].n_unique():,}")
print(f"Unique previous applications: {previous_df['SK_ID_PREV'].n_unique():,}")
print(f"Total application records: {previous_df.shape[0]:,}")
print(f"Average applications per customer: {previous_df.shape[0] / previous_df['SK_ID_CURR'].n_unique():.1f}")

# Missing values analysis
print(f"\nMissing Values Analysis:")
missing_analysis = []
for col in previous_df.columns:
    null_count = previous_df[col].null_count()
    null_pct = (null_count / previous_df.shape[0]) * 100
    missing_analysis.append({'column': col, 'null_count': null_count, 'null_percentage': null_pct})

missing_df = pl.DataFrame(missing_analysis).sort('null_percentage', descending=True)
print(missing_df.filter(pl.col('null_count') > 0))

In [ ]:
# === KEY BUSINESS PATTERNS ANALYSIS ===
print("\n=== PREVIOUS APPLICATION BUSINESS PATTERNS ===")

# 1. Contract Status Distribution (CRITICAL for risk)
print("1. CONTRACT STATUS DISTRIBUTION:")
status_counts = previous_df['NAME_CONTRACT_STATUS'].value_counts().sort('count',
descending=True)
print(status_counts)

# 2. Contract Type Analysis
print("\n2. CONTRACT TYPE ANALYSIS:")
contract_type_counts = previous_df['NAME_CONTRACT_TYPE'].value_counts().sort('count',
descending=True)
print(contract_type_counts)

# 3. Application vs Credit Amount Analysis (approval patterns)
print("\n3. APPLICATION vs CREDIT AMOUNT ANALYSIS:")
amount_analysis = previous_df.select([
    pl.col('AMT_APPLICATION').mean().alias('avg_requested'),
    pl.col('AMT_CREDIT').mean().alias('avg_approved'),
    pl.col('AMT_APPLICATION').median().alias('median_requested'),
    pl.col('AMT_CREDIT').median().alias('median_approved'),
    (pl.col('AMT_CREDIT') /
pl.col('AMT_APPLICATION')).mean().alias('avg_approval_ratio'),
    (pl.col('AMT_CREDIT') >
pl.col('AMT_APPLICATION')).sum().alias('over_approved_count'),
    (pl.col('AMT_CREDIT') <
pl.col('AMT_APPLICATION')).sum().alias('under_approved_count')
])
print(amount_analysis)


In [ ]:
# 4. Rejection Reason Analysis
print("\n4. REJECTION REASON ANALYSIS:")
rejection_analysis = previous_df.filter(
    pl.col('NAME_CONTRACT_STATUS') != 'Approved'
)['CODE_REJECT_REASON'].value_counts().sort('count', descending=True)
print(rejection_analysis.head(10))


# 5. Portfolio and Product Analysis
print("\n5. PORTFOLIO DISTRIBUTION:")
portfolio_counts = previous_df['NAME_PORTFOLIO'].value_counts().sort('count',
descending=True)
print(portfolio_counts)

# 6. Client Type Analysis (New vs Repeater)
print("\n6. CLIENT TYPE ANALYSIS:")
client_type_analysis = previous_df.group_by('NAME_CLIENT_TYPE').agg([
    pl.count().alias('application_count'),
    (pl.col('NAME_CONTRACT_STATUS') == 'Approved').sum().alias('approved_count'),
    (pl.col('NAME_CONTRACT_STATUS') == 'Approved').mean().alias('approval_rate')
])
print(client_type_analysis)


In [ ]:
# === ADVANCED PREVIOUS APPLICATION FEATURE ENGINEERING ===
print("=== PREVIOUS APPLICATION FEATURE ENGINEERING ===")

# Create comprehensive features at SK_ID_CURR level
previous_features = (
    previous_df
    .group_by("SK_ID_CURR")
    .agg([
        # === BASIC APPLICATION METRICS ===
        pl.count().alias("PREV_APP_COUNT"),
        pl.col("SK_ID_PREV").n_unique().alias("PREV_UNIQUE_APP_COUNT"),

        # === APPROVAL PATTERNS (CRITICAL RISK INDICATORS) ===
        (pl.col("NAME_CONTRACT_STATUS") == "Approved").sum().alias("PREV_APPROVED_COUNT"),
        (pl.col("NAME_CONTRACT_STATUS") == "Refused").sum().alias("PREV_REFUSED_COUNT"),
        (pl.col("NAME_CONTRACT_STATUS") == "Canceled").sum().alias("PREV_CANCELED_COUNT"),

        # Approval rates (key risk metric)
        ((pl.col("NAME_CONTRACT_STATUS") == "Approved").sum() / pl.count()).alias("PREV_APPROVAL_RATE"),
        ((pl.col("NAME_CONTRACT_STATUS") == "Refused").sum() / pl.count()).alias("PREV_REFUSAL_RATE"),

        # === CREDIT AMOUNT PATTERNS ===
        pl.col("AMT_APPLICATION").sum().alias("PREV_AMT_APPLICATION_TOTAL"),
        pl.col("AMT_APPLICATION").mean().alias("PREV_AMT_APPLICATION_MEAN"),
        pl.col("AMT_APPLICATION").max().alias("PREV_AMT_APPLICATION_MAX"),
        pl.col("AMT_APPLICATION").std().alias("PREV_AMT_APPLICATION_STD"),

        pl.col("AMT_CREDIT").sum().alias("PREV_AMT_CREDIT_TOTAL"),
        pl.col("AMT_CREDIT").mean().alias("PREV_AMT_CREDIT_MEAN"),
        pl.col("AMT_CREDIT").max().alias("PREV_AMT_CREDIT_MAX"),

        # Credit vs Application ratios (approval behavior)
        (pl.col("AMT_CREDIT") / pl.col("AMT_APPLICATION")).mean().alias("PREV_CREDIT_TO_APP_RATIO_MEAN"),
        (pl.col("AMT_CREDIT") / pl.col("AMT_APPLICATION")).min().alias("PREV_CREDIT_TO_APP_RATIO_MIN"),
        (pl.col("AMT_CREDIT") / pl.col("AMT_APPLICATION")).max().alias("PREV_CREDIT_TO_APP_RATIO_MAX"),

        # Over/under approval patterns
        (pl.col("AMT_CREDIT") > pl.col("AMT_APPLICATION")).sum().alias("PREV_OVER_APPROVED_COUNT"),
        (pl.col("AMT_CREDIT") < pl.col("AMT_APPLICATION")).sum().alias("PREV_UNDER_APPROVED_COUNT"),
        ((pl.col("AMT_CREDIT") > pl.col("AMT_APPLICATION")).sum() / pl.count()).alias("PREV_OVER_APPROVAL_RATIO"),

        # === ANNUITY PATTERNS ===
        pl.col("AMT_ANNUITY").mean().alias("PREV_ANNUITY_MEAN"),
        pl.col("AMT_ANNUITY").max().alias("PREV_ANNUITY_MAX"),
        (pl.col("AMT_ANNUITY") / pl.col("AMT_CREDIT")).mean().alias("PREV_ANNUITY_TO_CREDIT_RATIO"),

        # === DOWN PAYMENT ANALYSIS ===
        pl.col("AMT_DOWN_PAYMENT").sum().alias("PREV_DOWN_PAYMENT_TOTAL"),
        pl.col("AMT_DOWN_PAYMENT").mean().alias("PREV_DOWN_PAYMENT_MEAN"),
        (pl.col("AMT_DOWN_PAYMENT") > 0).sum().alias("PREV_WITH_DOWN_PAYMENT_COUNT"),
        ((pl.col("AMT_DOWN_PAYMENT") > 0).sum() / pl.count()).alias("PREV_DOWN_PAYMENT_RATIO"),
        (pl.col("AMT_DOWN_PAYMENT") / pl.col("AMT_GOODS_PRICE")).mean().alias("PREV_DOWN_PAYMENT_RATE_MEAN"),

        # === INTEREST RATE PATTERNS ===
        pl.col("RATE_INTEREST_PRIMARY").mean().alias("PREV_INTEREST_RATE_MEAN"),
        pl.col("RATE_INTEREST_PRIMARY").max().alias("PREV_INTEREST_RATE_MAX"),
        pl.col("RATE_INTEREST_PRIMARY").std().alias("PREV_INTEREST_RATE_STD"),

        # === TEMPORAL PATTERNS ===
        pl.col("DAYS_DECISION").mean().alias("PREV_DAYS_DECISION_MEAN"),
        pl.col("DAYS_DECISION").min().alias("PREV_DAYS_DECISION_MIN"),  # Most recent
        pl.col("DAYS_DECISION").max().alias("PREV_DAYS_DECISION_MAX"),  # Oldest

        # Time-based risk indicators
        (pl.col("DAYS_DECISION") > -30).sum().alias("PREV_RECENT_APP_COUNT"),   # Apps in last 30 days
        (pl.col("DAYS_DECISION") > -90).sum().alias("PREV_RECENT_90D_COUNT"),
        (pl.col("DAYS_DECISION") > -365).sum().alias("PREV_RECENT_1Y_COUNT"),

        # === CONTRACT TYPE PATTERNS ===
        (pl.col("NAME_CONTRACT_TYPE") == "Cash loans").sum().alias("PREV_CASH_LOAN_COUNT"),
        (pl.col("NAME_CONTRACT_TYPE") == "Consumer loans").sum().alias("PREV_CONSUMER_LOAN_COUNT"),
        (pl.col("NAME_CONTRACT_TYPE") == "Revolving loans").sum().alias("PREV_REVOLVING_LOAN_COUNT"),

        # === PORTFOLIO DIVERSIFICATION ===
        pl.col("NAME_PORTFOLIO").n_unique().alias("PREV_PORTFOLIO_DIVERSITY"),
        (pl.col("NAME_PORTFOLIO") == "Cash").sum().alias("PREV_CASH_PORTFOLIO_COUNT"),
        (pl.col("NAME_PORTFOLIO") == "POS").sum().alias("PREV_POS_PORTFOLIO_COUNT"),
        (pl.col("NAME_PORTFOLIO") == "Cards").sum().alias("PREV_CARDS_PORTFOLIO_COUNT"),

        # === CLIENT BEHAVIOR PATTERNS ===
        (pl.col("NAME_CLIENT_TYPE") == "Repeater").sum().alias("PREV_REPEATER_APP_COUNT"),
        (pl.col("NAME_CLIENT_TYPE") == "New").sum().alias("PREV_NEW_CLIENT_APP_COUNT"),
        ((pl.col("NAME_CLIENT_TYPE") == "Repeater").sum() / pl.count()).alias("PREV_REPEATER_RATIO"),

        # === PAYMENT METHOD PREFERENCES ===
        pl.col("NAME_PAYMENT_TYPE").n_unique().alias("PREV_PAYMENT_TYPE_DIVERSITY"),
        (pl.col("NAME_PAYMENT_TYPE") == "Cash through the bank").sum().alias("PREV_BANK_PAYMENT_COUNT"),

        # === CHANNEL ANALYSIS ===
        pl.col("CHANNEL_TYPE").n_unique().alias("PREV_CHANNEL_DIVERSITY"),

        # === GOODS AND PURPOSE ANALYSIS ===
        pl.col("NAME_GOODS_CATEGORY").n_unique().alias("PREV_GOODS_DIVERSITY"),
        pl.col("NAME_CASH_LOAN_PURPOSE").n_unique().alias("PREV_PURPOSE_DIVERSITY"),

        # === ADVANCED RISK INDICATORS ===
        (pl.col("NFLAG_LAST_APPL_IN_DAY") == 0).sum().alias("PREV_MULTIPLE_SAME_DAY_COUNT"),
        pl.col("NFLAG_INSURED_ON_APPROVAL").mean().alias("PREV_INSURANCE_RATIO"),
        (pl.col("NAME_YIELD_GROUP") == "high").sum().alias("PREV_HIGH_YIELD_COUNT"),
        (pl.col("NAME_YIELD_GROUP") == "low_action").sum().alias("PREV_LOW_ACTION_COUNT"),

        # === REJECTION ANALYSIS ===
        (pl.col("CODE_REJECT_REASON") == "HC").sum().alias("PREV_HC_REJECT_COUNT"),       # High credit risk
        (pl.col("CODE_REJECT_REASON") == "LIMIT").sum().alias("PREV_LIMIT_REJECT_COUNT"),
        (pl.col("CODE_REJECT_REASON") == "SCO").sum().alias("PREV_SCO_REJECT_COUNT"),     # Scoring

        # === PAYMENT TERMS ANALYSIS ===
        pl.col("CNT_PAYMENT").mean().alias("PREV_PAYMENT_TERMS_MEAN"),
        pl.col("CNT_PAYMENT").max().alias("PREV_PAYMENT_TERMS_MAX"),
    ])
)

print(f"Previous application features shape: {previous_features.shape}")
print(f"Features created: {len(previous_features.columns) - 1}")


In [ ]:
# === ADVANCED: CROSS-TABLE LINKAGE FEATURES ===
print("\n=== CROSS-TABLE LINKAGE ANALYSIS ===")

# NOTE: These features would require loading and joining with other tables
# For now, we'll create the framework and identify the linkage opportunities

print("""
🔗 COMPLEX LINKAGE OPPORTUNITIES:

1. PREVIOUS APPLICATION ↔ POS_CASH_BALANCE:
   - Link via SK_ID_PREV to get repayment behavior for previous loans
   - Potential features:
     • PREV_POS_DPD_PATTERNS: Historical delinquency on POS loans
     • PREV_POS_PAYMENT_BEHAVIOR: Consistency of payments
     • PREV_POS_COMPLETION_RATE: How often loans were fully repaid

2. PREVIOUS APPLICATION ↔ INSTALLMENTS_PAYMENTS:
   - Link via SK_ID_PREV to get detailed payment history
   - Potential features:
     • PREV_PAYMENT_PUNCTUALITY: On-time payment rate
     • PREV_OVERPAYMENT_BEHAVIOR: Tendency to pay more than required
     • PREV_PAYMENT_CONSISTENCY: Variability in payment amounts

3. PREVIOUS APPLICATION ↔ CREDIT_CARD_BALANCE:
   - Link via SK_ID_PREV for credit card previous loans
   - Potential features:
     • PREV_CC_UTILIZATION_PATTERNS: Historical credit card usage
     • PREV_CC_PAYMENT_BEHAVIOR: Credit card payment discipline
     • PREV_CC_LIMIT_MANAGEMENT: How well they managed limits

🎯 IMPLEMENTATION STRATEGY:
   1. Start with Previous Application standalone features (current approach)
   2. Add cross-table features incrementally for maximum impact
   3. Focus on approved previous loans first (they have payment history)
   4. Prioritize recent previous loans (more predictive of current behavior)
""")

# Create a flag for approved applications that can be linked to payment history
previous_features_with_linkage = (
    previous_df
    .group_by("SK_ID_CURR")
    .agg([
        # Previous basic features (abbreviated list)
        pl.count().alias("PREV_APP_COUNT"),
        (pl.col("NAME_CONTRACT_STATUS") == "Approved").sum().alias("PREV_APPROVED_COUNT"),
        ((pl.col("NAME_CONTRACT_STATUS") == "Approved").sum() / pl.count()).alias("PREV_APPROVAL_RATE"),

        # === LINKAGE-READY FEATURES ===
        # Count approved apps by type (for linkage to payment tables)
        (
            (pl.col("NAME_CONTRACT_STATUS") == "Approved") &
            (pl.col("NAME_PORTFOLIO") == "POS")
        ).sum().alias("PREV_APPROVED_POS_COUNT"),

        (
            (pl.col("NAME_CONTRACT_STATUS") == "Approved") &
            (pl.col("NAME_PORTFOLIO") == "Cash")
        ).sum().alias("PREV_APPROVED_CASH_COUNT"),

        (
            (pl.col("NAME_CONTRACT_STATUS") == "Approved") &
            (pl.col("NAME_PORTFOLIO") == "Cards")
        ).sum().alias("PREV_APPROVED_CARDS_COUNT"),

        # Recent approved applications (with likely payment history)
        (
            (pl.col("NAME_CONTRACT_STATUS") == "Approved") &
            (pl.col("DAYS_DECISION") > -365)
        ).sum().alias("PREV_RECENT_APPROVED_COUNT"),

        # Store SK_ID_PREV for approved loans (list column for future linking)
        pl.col("SK_ID_PREV")
          .filter(pl.col("NAME_CONTRACT_STATUS") == "Approved")
          .alias("PREV_APPROVED_LOAN_IDS"),

        # Most recent approved loan details
        pl.col("AMT_CREDIT")
          .filter(pl.col("NAME_CONTRACT_STATUS") == "Approved")
          .max()
          .alias("PREV_LAST_APPROVED_AMOUNT"),

        pl.col("DAYS_DECISION")
          .filter(pl.col("NAME_CONTRACT_STATUS") == "Approved")
          .min()
          .alias("PREV_LAST_APPROVED_DAYS"),  # Most recent (closest to 0)
    ])
)

print(f"\nLinkage-ready features created: {previous_features_with_linkage.shape}")

# === IMMEDIATE VALUE FEATURES (NO LINKAGE NEEDED) ===
print("\n=== HIGH-IMPACT STANDALONE FEATURES ===")

# Focus on the most predictive features that don't need cross-table joins
high_impact_features = (
    previous_df
    .group_by("SK_ID_CURR")
    .agg([
        # === CORE RISK INDICATORS ===
        pl.count().alias("PREV_APP_COUNT"),
        ((pl.col("NAME_CONTRACT_STATUS") == "Approved").sum() / pl.count()).alias("PREV_APPROVAL_RATE"),
        ((pl.col("NAME_CONTRACT_STATUS") == "Refused").sum() / pl.count()).alias("PREV_REFUSAL_RATE"),

        # === CREDIT BEHAVIOR ===
        (pl.col("AMT_CREDIT") / pl.col("AMT_APPLICATION")).mean().alias("PREV_APPROVAL_RATIO_MEAN"),
        pl.col("AMT_CREDIT").sum().alias("PREV_TOTAL_CREDIT_HISTORY"),
        pl.col("AMT_APPLICATION").sum().alias("PREV_TOTAL_REQUESTED_HISTORY"),

        # === RISK FLAGS ===
        (pl.col("CODE_REJECT_REASON") == "HC").sum().alias("PREV_HIGH_RISK_REJECTS"),
        (pl.col("RATE_INTEREST_PRIMARY") > 0.2).sum().alias("PREV_HIGH_INTEREST_COUNT"),
        ((pl.col("AMT_DOWN_PAYMENT") == 0) | pl.col("AMT_DOWN_PAYMENT").is_null()).sum().alias("PREV_NO_DOWN_PAYMENT_COUNT"),

        # === BEHAVIORAL PATTERNS ===
        (pl.col("NAME_CLIENT_TYPE") == "Repeater").sum().alias("PREV_REPEAT_CUSTOMER_APPS"),
        pl.col("NAME_PORTFOLIO").n_unique().alias("PREV_PRODUCT_DIVERSITY"),
        pl.col("DAYS_DECISION").min().alias("PREV_MOST_RECENT_APP_DAYS"),

        # === VOLUME AND FREQUENCY INDICATORS ===
        (pl.col("DAYS_DECISION") > -90).sum().alias("PREV_APPS_LAST_90_DAYS"),
        (pl.col("DAYS_DECISION") > -365).sum().alias("PREV_APPS_LAST_YEAR"),
    ])
)

print(f"High-impact standalone features: {high_impact_features.shape}")

print("\n🎯 NEXT STEPS:")
print("1. Implement standalone features first (immediate value)")
print("2. Save previous application features to CSV")
print("3. Test model improvement with these features")
print("4. Later: Add cross-table linkage features for advanced modeling")


In [ ]:
# === COMPREHENSIVE PREVIOUS APPLICATION FEATURE ENGINEERING ===
print("=== CREATING COMPREHENSIVE PREVIOUS APPLICATION FEATURES ===")

# Create the full feature set at SK_ID_CURR level
previous_application_features = (
    previous_df
    .group_by("SK_ID_CURR")
    .agg([
        # === BASIC APPLICATION METRICS ===
        pl.count().alias("PREV_APP_COUNT"),
        pl.col("SK_ID_PREV").n_unique().alias("PREV_UNIQUE_APP_COUNT"),

        # === APPROVAL PATTERNS (CRITICAL RISK INDICATORS) ===
        (pl.col("NAME_CONTRACT_STATUS") == "Approved").sum().alias("PREV_APPROVED_COUNT"),
        (pl.col("NAME_CONTRACT_STATUS") == "Refused").sum().alias("PREV_REFUSED_COUNT"),
        (pl.col("NAME_CONTRACT_STATUS") == "Canceled").sum().alias("PREV_CANCELED_COUNT"),

        # Approval rates (key risk metrics)
        ((pl.col("NAME_CONTRACT_STATUS") == "Approved").sum() / pl.count()).alias("PREV_APPROVAL_RATE"),
        ((pl.col("NAME_CONTRACT_STATUS") == "Refused").sum() / pl.count()).alias("PREV_REFUSAL_RATE"),
        ((pl.col("NAME_CONTRACT_STATUS") == "Canceled").sum() / pl.count()).alias("PREV_CANCEL_RATE"),

        # === CREDIT AMOUNT PATTERNS ===
        pl.col("AMT_APPLICATION").sum().alias("PREV_AMT_APPLICATION_TOTAL"),
        pl.col("AMT_APPLICATION").mean().alias("PREV_AMT_APPLICATION_MEAN"),
        pl.col("AMT_APPLICATION").max().alias("PREV_AMT_APPLICATION_MAX"),
        pl.col("AMT_APPLICATION").std().alias("PREV_AMT_APPLICATION_STD"),

        pl.col("AMT_CREDIT").sum().alias("PREV_AMT_CREDIT_TOTAL"),
        pl.col("AMT_CREDIT").mean().alias("PREV_AMT_CREDIT_MEAN"),
        pl.col("AMT_CREDIT").max().alias("PREV_AMT_CREDIT_MAX"),
        pl.col("AMT_CREDIT").std().alias("PREV_AMT_CREDIT_STD"),

        # Credit vs Application ratios
        (pl.col("AMT_CREDIT") / pl.col("AMT_APPLICATION")).mean().alias("PREV_CREDIT_TO_APP_RATIO_MEAN"),
        (pl.col("AMT_CREDIT") / pl.col("AMT_APPLICATION")).min().alias("PREV_CREDIT_TO_APP_RATIO_MIN"),
        (pl.col("AMT_CREDIT") / pl.col("AMT_APPLICATION")).max().alias("PREV_CREDIT_TO_APP_RATIO_MAX"),
        (pl.col("AMT_CREDIT") / pl.col("AMT_APPLICATION")).std().alias("PREV_CREDIT_TO_APP_RATIO_STD"),

        # Over/under approval patterns
        (pl.col("AMT_CREDIT") > pl.col("AMT_APPLICATION")).sum().alias("PREV_OVER_APPROVED_COUNT"),
        (pl.col("AMT_CREDIT") < pl.col("AMT_APPLICATION")).sum().alias("PREV_UNDER_APPROVED_COUNT"),
        ((pl.col("AMT_CREDIT") > pl.col("AMT_APPLICATION")).sum() / pl.count()).alias("PREV_OVER_APPROVAL_RATIO"),

        # === ANNUITY PATTERNS ===
        pl.col("AMT_ANNUITY").mean().alias("PREV_ANNUITY_MEAN"),
        pl.col("AMT_ANNUITY").max().alias("PREV_ANNUITY_MAX"),
        pl.col("AMT_ANNUITY").std().alias("PREV_ANNUITY_STD"),
        (pl.col("AMT_ANNUITY") / pl.col("AMT_CREDIT")).mean().alias("PREV_ANNUITY_TO_CREDIT_RATIO"),

        # === DOWN PAYMENT ANALYSIS ===
        pl.col("AMT_DOWN_PAYMENT").sum().alias("PREV_DOWN_PAYMENT_TOTAL"),
        pl.col("AMT_DOWN_PAYMENT").mean().alias("PREV_DOWN_PAYMENT_MEAN"),
        pl.col("AMT_DOWN_PAYMENT").max().alias("PREV_DOWN_PAYMENT_MAX"),
        (pl.col("AMT_DOWN_PAYMENT") > 0).sum().alias("PREV_WITH_DOWN_PAYMENT_COUNT"),
        ((pl.col("AMT_DOWN_PAYMENT") > 0).sum() / pl.count()).alias("PREV_DOWN_PAYMENT_RATIO"),
        pl.col("RATE_DOWN_PAYMENT").mean().alias("PREV_DOWN_PAYMENT_RATE_MEAN"),

        # === INTEREST RATE PATTERNS ===
        pl.col("RATE_INTEREST_PRIMARY").mean().alias("PREV_INTEREST_RATE_MEAN"),
        pl.col("RATE_INTEREST_PRIMARY").max().alias("PREV_INTEREST_RATE_MAX"),
        pl.col("RATE_INTEREST_PRIMARY").min().alias("PREV_INTEREST_RATE_MIN"),
        pl.col("RATE_INTEREST_PRIMARY").std().alias("PREV_INTEREST_RATE_STD"),

        # === TEMPORAL PATTERNS ===
        pl.col("DAYS_DECISION").mean().alias("PREV_DAYS_DECISION_MEAN"),
        pl.col("DAYS_DECISION").min().alias("PREV_DAYS_DECISION_MIN"),  # Most recent
        pl.col("DAYS_DECISION").max().alias("PREV_DAYS_DECISION_MAX"),  # Oldest
        pl.col("DAYS_DECISION").std().alias("PREV_DAYS_DECISION_STD"),

        # Time-based risk indicators
        (pl.col("DAYS_DECISION") > -30).sum().alias("PREV_APPS_LAST_30D"),
        (pl.col("DAYS_DECISION") > -90).sum().alias("PREV_APPS_LAST_90D"),
        (pl.col("DAYS_DECISION") > -365).sum().alias("PREV_APPS_LAST_1Y"),
        (pl.col("DAYS_DECISION") > -730).sum().alias("PREV_APPS_LAST_2Y"),

        # Recent approval patterns
        ((pl.col("DAYS_DECISION") > -365) & (pl.col("NAME_CONTRACT_STATUS") == "Approved")).sum().alias("PREV_RECENT_APPROVED_1Y"),
        ((pl.col("DAYS_DECISION") > -90) & (pl.col("NAME_CONTRACT_STATUS") == "Approved")).sum().alias("PREV_RECENT_APPROVED_90D"),

        # === CONTRACT TYPE PATTERNS ===
        (pl.col("NAME_CONTRACT_TYPE") == "Cash loans").sum().alias("PREV_CASH_LOAN_COUNT"),
        (pl.col("NAME_CONTRACT_TYPE") == "Consumer loans").sum().alias("PREV_CONSUMER_LOAN_COUNT"),
        (pl.col("NAME_CONTRACT_TYPE") == "Revolving loans").sum().alias("PREV_REVOLVING_LOAN_COUNT"),
        ((pl.col("NAME_CONTRACT_TYPE") == "Cash loans").sum() / pl.count()).alias("PREV_CASH_LOAN_RATIO"),

        # === PORTFOLIO DIVERSIFICATION ===
        pl.col("NAME_PORTFOLIO").n_unique().alias("PREV_PORTFOLIO_DIVERSITY"),
        (pl.col("NAME_PORTFOLIO") == "Cash").sum().alias("PREV_CASH_PORTFOLIO_COUNT"),
        (pl.col("NAME_PORTFOLIO") == "POS").sum().alias("PREV_POS_PORTFOLIO_COUNT"),
        (pl.col("NAME_PORTFOLIO") == "Cards").sum().alias("PREV_CARDS_PORTFOLIO_COUNT"),
        ((pl.col("NAME_PORTFOLIO") == "Cash").sum() / pl.count()).alias("PREV_CASH_PORTFOLIO_RATIO"),

        # === CLIENT BEHAVIOR PATTERNS ===
        (pl.col("NAME_CLIENT_TYPE") == "Repeater").sum().alias("PREV_REPEATER_APP_COUNT"),
        (pl.col("NAME_CLIENT_TYPE") == "New").sum().alias("PREV_NEW_CLIENT_APP_COUNT"),
        ((pl.col("NAME_CLIENT_TYPE") == "Repeater").sum() / pl.count()).alias("PREV_REPEATER_RATIO"),

        # === CHANNEL AND PAYMENT ANALYSIS ===
        pl.col("CHANNEL_TYPE").n_unique().alias("PREV_CHANNEL_DIVERSITY"),
        pl.col("NAME_PAYMENT_TYPE").n_unique().alias("PREV_PAYMENT_TYPE_DIVERSITY"),
        (pl.col("NAME_PAYMENT_TYPE") == "Cash through the bank").sum().alias("PREV_BANK_PAYMENT_COUNT"),

        # === GOODS AND PURPOSE ANALYSIS ===
        pl.col("NAME_GOODS_CATEGORY").n_unique().alias("PREV_GOODS_DIVERSITY"),
        pl.col("NAME_CASH_LOAN_PURPOSE").n_unique().alias("PREV_PURPOSE_DIVERSITY"),

        # === ADVANCED RISK INDICATORS ===
        (pl.col("NFLAG_LAST_APPL_IN_DAY") == 0).sum().alias("PREV_MULTIPLE_SAME_DAY_COUNT"),
        pl.col("NFLAG_INSURED_ON_APPROVAL").mean().alias("PREV_INSURANCE_RATIO"),
        (pl.col("NAME_YIELD_GROUP") == "high").sum().alias("PREV_HIGH_YIELD_COUNT"),
        (pl.col("NAME_YIELD_GROUP") == "low_action").sum().alias("PREV_LOW_ACTION_COUNT"),
        (pl.col("NAME_YIELD_GROUP") == "middle").sum().alias("PREV_MIDDLE_YIELD_COUNT"),

        # === REJECTION ANALYSIS ===
        (pl.col("CODE_REJECT_REASON") == "HC").sum().alias("PREV_HC_REJECT_COUNT"),       # High credit risk
        (pl.col("CODE_REJECT_REASON") == "LIMIT").sum().alias("PREV_LIMIT_REJECT_COUNT"), # Limit exceeded
        (pl.col("CODE_REJECT_REASON") == "SCO").sum().alias("PREV_SCO_REJECT_COUNT"),     # Scoring
        (pl.col("CODE_REJECT_REASON") == "CLIENT").sum().alias("PREV_CLIENT_REJECT_COUNT"), # Client issues

        # === PAYMENT TERMS ANALYSIS ===
        pl.col("CNT_PAYMENT").mean().alias("PREV_PAYMENT_TERMS_MEAN"),
        pl.col("CNT_PAYMENT").max().alias("PREV_PAYMENT_TERMS_MAX"),
        pl.col("CNT_PAYMENT").min().alias("PREV_PAYMENT_TERMS_MIN"),
        pl.col("CNT_PAYMENT").std().alias("PREV_PAYMENT_TERMS_STD"),

        # === GOODS PRICE ANALYSIS ===
        pl.col("AMT_GOODS_PRICE").mean().alias("PREV_GOODS_PRICE_MEAN"),
        pl.col("AMT_GOODS_PRICE").max().alias("PREV_GOODS_PRICE_MAX"),
        (pl.col("AMT_GOODS_PRICE") / pl.col("AMT_CREDIT")).mean().alias("PREV_GOODS_TO_CREDIT_RATIO"),
    ])
)

print(f"Previous application features shape: {previous_application_features.shape}")
print(f"Features created: {len(previous_application_features.columns) - 1}")  # -1 for SK_ID_CURR


In [ ]:
# === FEATURE QUALITY VALIDATION ===
print("\n=== PREVIOUS APPLICATION FEATURE QUALITY VALIDATION ===")

# Check feature completeness and data quality
print(f"Total customers with previous application history: {previous_application_features.shape[0]:,}")
print(f"Total features created: {len(previous_application_features.columns) - 1}")

# Missing value analysis for created features
print("\nMissing values in engineered features:")
missing_features = []
for col in previous_application_features.columns:
    if col != "SK_ID_CURR":
        null_count = previous_application_features[col].null_count()
        if null_count > 0:
            missing_features.append({
                "feature": col,
                "null_count": null_count,
                "null_pct": (null_count / previous_application_features.shape[0]) * 100
            })

if missing_features:
    missing_df = pl.DataFrame(missing_features).sort("null_pct", descending=True)
    print(missing_df.head(10))
else:
    print("✅ No missing values in engineered features!")

# Infinite/extreme value checks
print("\nChecking for infinite or extreme values:")
extreme_features = []
for col in previous_application_features.columns:
    if col != "SK_ID_CURR" and previous_application_features[col].dtype in [pl.Float64, pl.Float32]:
        col_stats = previous_application_features.select([
            pl.col(col).min().alias("min"),
            pl.col(col).max().alias("max"),
            pl.col(col).is_infinite().sum().alias("inf_count")
        ]).row(0)

        if col_stats[2] > 0 or col_stats[0] < -1e10 or col_stats[1] > 1e10:
            extreme_features.append({
                "feature": col,
                "min": col_stats[0],
                "max": col_stats[1],
                "inf_count": col_stats[2]
            })

if extreme_features:
    print("⚠️  Features with extreme/infinite values:")
    for feat in extreme_features[:10]:
        print(
            f"   {feat['feature']}: "
            f"min={feat['min']:.2e}, "
            f"max={feat['max']:.2e}, "
            f"inf_count={feat['inf_count']}"
        )
else:
    print("✅ No extreme or infinite values detected!")

# Feature distribution summary for key risk indicators
print("\nKey risk indicator distributions:")
key_risk_features = [
    "PREV_APPROVAL_RATE",
    "PREV_REFUSAL_RATE",
    "PREV_CREDIT_TO_APP_RATIO_MEAN",
    "PREV_HC_REJECT_COUNT",
    "PREV_REPEATER_RATIO"
]

for feat in key_risk_features:
    if feat in previous_application_features.columns:
        stats = previous_application_features.select([
            pl.col(feat).mean().alias("mean"),
            pl.col(feat).std().alias("std"),
            pl.col(feat).quantile(0.5).alias("median"),
            pl.col(feat).quantile(0.95).alias("p95")
        ]).row(0)

        print(
            f"📊 {feat}: "
            f"mean={stats[0]:.3f}, "
            f"std={stats[1]:.3f}, "
            f"median={stats[2]:.3f}, "
            f"p95={stats[3]:.3f}"
        )


In [ ]:
# === CLEAN AND SAVE PREVIOUS APPLICATION FEATURES ===
print("\n=== SAVING PREVIOUS APPLICATION FEATURES ===")

# Clean infinite values before saving
previous_application_features_clean = previous_application_features.with_columns([
    pl.when(pl.col(col).is_infinite())
      .then(None)
      .otherwise(pl.col(col))
      .alias(col)
    for col in previous_application_features.columns
    if col != "SK_ID_CURR"
])

# Save to CSV for integration with main model
output_path = "../data/previous_application_features.csv"
previous_application_features_clean.write_csv(output_path)

print(f"✅ Previous application features saved to: {output_path}")
print(f"Shape: {previous_application_features_clean.shape}")

# Feature summary for documentation
print("\n📋 PREVIOUS APPLICATION FEATURE SUMMARY:")
print(f"Total customers with previous application history: {previous_application_features_clean.shape[0]:,}")
print(f"Total features engineered: {len(previous_application_features_clean.columns) - 1}")

# Feature importance preview (based on business logic)
high_importance_features = [
    "PREV_APPROVAL_RATE", "PREV_REFUSAL_RATE", "PREV_CREDIT_TO_APP_RATIO_MEAN",
    "PREV_HC_REJECT_COUNT", "PREV_REPEATER_RATIO", "PREV_APPS_LAST_1Y",
    "PREV_RECENT_APPROVED_1Y", "PREV_OVER_APPROVAL_RATIO", "PREV_INTEREST_RATE_MEAN",
    "PREV_MULTIPLE_SAME_DAY_COUNT", "PREV_DOWN_PAYMENT_RATIO"
]

print("\n🎯 HIGH-IMPACT RISK FEATURES (ready for model integration):")
for feat in high_importance_features:
    if feat in previous_application_features_clean.columns:
        print(f"   • {feat}")

# Feature categories breakdown
feature_categories = {
    "Approval Behavior": len([
        col for col in previous_application_features_clean.columns
        if "APPROVAL" in col or "REFUSED" in col or "APPROVED" in col
    ]),
    "Credit Amounts": len([
        col for col in previous_application_features_clean.columns
        if "AMT_CREDIT" in col or "AMT_APPLICATION" in col
    ]),
    "Risk Indicators": len([
        col for col in previous_application_features_clean.columns
        if "REJECT" in col or "RATIO" in col or "RATE" in col
    ]),
    "Temporal Patterns": len([
        col for col in previous_application_features_clean.columns
        if "DAYS" in col or "RECENT" in col or "LAST" in col
    ]),
    "Product Portfolio": len([
        col for col in previous_application_features_clean.columns
        if "PORTFOLIO" in col or "CONTRACT" in col or "LOAN" in col
    ]),
    "Behavioral": len([
        col for col in previous_application_features_clean.columns
        if "REPEATER" in col or "CLIENT" in col or "DIVERSITY" in col
    ]),
}

print("\n📊 FEATURE CATEGORIES:")
for category, count in feature_categories.items():
    print(f"   • {category}: {count} features")

print("\n💡 INTEGRATION INSTRUCTIONS:")
print("1. Merge with application_train on SK_ID_CURR")
print("2. Handle missing values (customers without previous application history)")
print("3. Add to existing baseline + POS + bureau + credit_card feature set")
print("4. Expected AUC improvement: +0.010 to +0.025 based on lending history patterns")
print("5. Key predictive signals: approval rates, credit ratios, rejection patterns, temporal behavior")
